In [ ]:
##Spatial Consistency Check

In [ ]:
#1.Representativeness of a Pair

In [ ]:
your-repo-name/
│
├── metadata/
│   └── station_elevation.csv      <-- (Input) Station list with Lat, Lon, Elev
├── output/
│   └── advanced_qc/
│       └── cleaned_daily_data/    <-- (Input) CSVs from the previous Q-cleaning step
│   └── spatial_check/             <-- (Output) Neighbor pairs will be saved here
└── spatial_step1_pairs.py         <-- This script

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from math import radians, sin, cos, asin, sqrt
import re

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: Metadata containing columns: station_id, lat, lon, elev_m
STATIONS_CSV = BASE_DIR / "metadata" / "station_elevation.csv"

# Input: Cleaned daily files (Output from previous Q-Index step)
DAILY_DIR = BASE_DIR / "output" / "advanced_qc" / "cleaned_daily_data"

# Output: Neighboring station pairs and R-scores
OUT_DIR = BASE_DIR / "output" / "spatial_check"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "spatial_neighbor_pairs.csv"

# Column Mapping
ID_COL = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# Calculation Thresholds
RAINY_DAY_THRESHOLD = 1.0     # Minimum rain to consider a "rainy day" (mm)
D_MAX_KM = 50.0               # Maximum search radius (Influence area)
R_MIN = 70.0                  # Threshold for auxiliary station suitability
MIN_JOINT_RAINY_DAYS = 25     # Min days with rain >= 1mm in both stations to compute correlation

# ==========================================
# Utilities
# ==========================================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Extracts date (YYYY-MM-DD) from filename."""
    s = path.stem
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", s)
    if not m:
        return pd.NaT
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def haversine_km(lat1, lon1, lat2, lon2):
    """Calculates the great-circle distance between two points in KM."""
    R = 6371.0088
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

def normalize_id(s: pd.Series) -> pd.Series:
    """Standardizes IDs to string format."""
    return (s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True))

# ==========================================
# 1. Load Station Metadata
# ==========================================
print("[+] Loading station metadata...")
st = pd.read_csv(STATIONS_CSV)
for col in [ID_COL, "lat", "lon", "elev_m"]:
    if col not in st.columns:
        raise ValueError(f"Metadata missing required column: '{col}'")

st[ID_COL] = normalize_id(st[ID_COL])
st = st[[ID_COL, "lat", "lon", "elev_m"]].copy()

# ==========================================
# 2. Load Daily Rain Data (Pivot to Wide format)
# ==========================================
print("[+] Processing daily files into wide format...")
rows = []
csv_files = sorted(DAILY_DIR.glob("*.csv"))

if not csv_files:
    raise RuntimeError(f"No daily CSV files found in: {DAILY_DIR}")

for fp in csv_files:
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        print(f" [!] Skipping {fp.name}: Missing required columns.")
        continue

    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})
    df["station_id"] = normalize_id(df["station_id"])

    # Extract date: From column first, fallback to filename
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)

    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan
    rows.append(df[["station_id", "date", "rain"]])

daily_long = pd.concat(rows, ignore_index=True).dropna(subset=["date"])
daily_wide = daily_long.pivot(index="date", columns="station_id", values="rain").sort_index()

# ==========================================
# 3. Identify Candidate Pairs within D_MAX
# ==========================================
print(f"[+] Finding pairs within {D_MAX_KM} km radius...")
pairs = []
for i, si in st.iterrows():
    for j, sj in st.iterrows():
        if si[ID_COL] == sj[ID_COL]:
            continue
        
        dist_km = haversine_km(si.lat, si.lon, sj.lat, sj.lon)
        if dist_km <= D_MAX_KM + 1e-9:
            alt_diff_m = abs(si.elev_m - sj.elev_m)
            pairs.append((si[ID_COL], sj[ID_COL], dist_km, alt_diff_m))

pairs_df = pd.DataFrame(pairs, columns=["cand_id", "aux_id", "dist_km", "alt_diff_m"])
if pairs_df.empty:
    raise RuntimeError("No station pairs found within the specified distance threshold.")

# ==========================================
# 4. Correlation and Joint Rainy Days
# ==========================================
print("[+] Computing correlation and joint rainy days...")

def get_stats(cand, aux):
    if cand not in daily_wide or aux not in daily_wide:
        return 0.0, 0
    
    # Align both series and drop missing days
    aligned = pd.concat([daily_wide[cand], daily_wide[aux]], axis=1).dropna()
    aligned.columns = ["c", "a"]
    
    # Count joint rainy days (both >= threshold)
    joint_rainy = int(((aligned["c"] >= RAINY_DAY_THRESHOLD) & 
                       (aligned["a"] >= RAINY_DAY_THRESHOLD)).sum())
    
    if joint_rainy < MIN_JOINT_RAINY_DAYS:
        return 0.0, joint_rainy
    
    # Pearson Correlation
    corr = aligned["c"].corr(aligned["a"])
    corr_val = float(max(0.0, min(1.0, corr))) if not pd.isna(corr) else 0.0
    return corr_val, joint_rainy

stats_results = []
for cand, aux in pairs_df[["cand_id", "aux_id"]].itertuples(index=False):
    c_val, n_joint = get_stats(cand, aux)
    stats_results.append((c_val, n_joint))

pairs_df[["ccorr", "joint_rainy_days"]] = stats_results

# ==========================================
# 5. Compute R-Index Score
# ==========================================
print("[+] Computing final R-scores...")

# Equation: (100/3) * (Term_Distance + Term_Altitude + Term_Correlation)
term_dist = ((D_MAX_KM - pairs_df["dist_km"]) / D_MAX_KM).clip(0, 1)
term_alt  = (0.5 ** (pairs_df["alt_diff_m"] / 500.0)).clip(0, 1)
term_corr = pairs_df["ccorr"].clip(0, 1)

r_score = (100.0 / 3.0) * (term_dist + term_alt + term_corr)

pairs_df["term_dist"] = term_dist
pairs_df["term_alt"] = term_alt
pairs_df["term_corr"] = term_corr
pairs_df["r_score"] = r_score.clip(0, 100)
pairs_df["is_suitable_aux"] = np.where(pairs_df["r_score"] >= R_MIN, "Yes", "No")

# ==========================================
# Save Output
# ==========================================
final_output = pairs_df.sort_values(["cand_id", "r_score"], ascending=[True, False])
final_output.to_csv(OUT_PATH, index=False)

print(f"\n[+] Neighbor analysis completed.")
print(f"    - Output saved to: {OUT_PATH}")
print(f"\nSample Data (Top 10):")
print(final_output.head(10))

In [ ]:
#2.Daily Difference and Monthly Threshold

In [ ]:
##2.1 Cm-Monthly Threshold

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: Neighbor pairs from Step 1 (containing cand_id, aux_id, r_score)
PAIRS_PATH = BASE_DIR / "output" / "spatial_check" / "spatial_neighbor_pairs.csv"

# Input: Directory with cleaned daily files (Q >= 50)
DAILY_DIR = BASE_DIR / "output" / "advanced_qc" / "cleaned_daily_data"

# Output: Month-specific threshold table (Cm)
OUT_DIR = BASE_DIR / "output" / "spatial_check"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CM_PATH = OUT_DIR / "spatial_cm_table.csv"

# Column Mapping
ID_COL = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# Threshold Settings
R_MIN = 70.0              # Minimum R-score to consider a neighbor pair
QTL_LEVEL = 0.95          # Quantile for Cm calculation (95th percentile)
MIN_POINTS_PER_MONTH = 30 # Min observations per month to use quantile (else use median)

# ==========================================
# Utilities
# ==========================================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Extracts date from filename."""
    s = path.stem
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", s)
    if not m:
        return pd.NaT
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def normalize_id(s: pd.Series) -> pd.Series:
    """Standardizes IDs to string format."""
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

# ==========================================
# 1. Load Neighbor Pairs
# ==========================================
print(f"[+] Loading neighbor pairs (R >= {R_MIN})...")
pairs = pd.read_csv(PAIRS_PATH)

# Rename to match Step 1 output if necessary
pairs = pairs.rename(columns={"cand_id": "cand", "aux_id": "aux", "r_score": "R"})

pairs["cand"] = normalize_id(pairs["cand"])
pairs["aux"] = normalize_id(pairs["aux"])

# Filter only reliable pairs
pairs_use = pairs[pairs["R"] >= R_MIN][["cand", "aux", "R"]].copy()

if pairs_use.empty:
    raise RuntimeError(f"No pairs found with R >= {R_MIN} in {PAIRS_PATH}")

# ==========================================
# 2. Load Daily Rain Data
# ==========================================
print("[+] Loading cleaned daily CSV files...")
rows = []
csv_files = sorted(DAILY_DIR.glob("*.csv"))

if not csv_files:
    raise RuntimeError(f"No daily CSV files found in: {DAILY_DIR}")

for fp in csv_files:
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        continue
        
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})
    df["station_id"] = normalize_id(df["station_id"])
    
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)
        
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan
    rows.append(df[["station_id", "date", "rain"]])

daily_df = pd.concat(rows, ignore_index=True).dropna(subset=["date"])
daily_df["date"] = pd.to_datetime(daily_df["date"])
daily_df["month"] = daily_df["date"].dt.month

# Group by station for fast lookup
by_sta = {sid: g.set_index("date")[["rain", "month"]].sort_index()
          for sid, g in daily_df.groupby("station_id")}

# ==========================================
# 3. Calculate Z-values for all Pairs
# ==========================================
print("[+] Computing Z-values (Difference/Penalty Term)...")
z_results = []

for cand, sub_pairs in pairs_use.groupby("cand"):
    s_c = by_sta.get(cand)
    if s_c is None:
        continue
        
    for aux, r_val in sub_pairs[["aux", "R"]].itertuples(index=False):
        s_a = by_sta.get(aux)
        if s_a is None:
            continue
            
        # Join Candidate and Auxiliary series
        s_a_rain = s_a[["rain"]].rename(columns={"rain": "rain_aux"})
        combined = s_c.join(s_a_rain, how="inner").dropna(subset=["rain", "rain_aux"])

        if combined.empty:
            continue
            
        ppt_c = combined["rain"].values
        ppt_a = combined["rain_aux"].values
        
        # DIF Calculation: (abs(c-a)/avg) * max
        avg_val = 0.5 * (ppt_c + ppt_a)
        max_val = np.maximum(ppt_c, ppt_a)
        
        # Avoid division by zero
        dif = np.where((ppt_c == 0) & (ppt_a == 0), 0.0, 
                       np.abs(ppt_c - ppt_a) / np.where(avg_val == 0, 1e-9, avg_val) * max_val)

        # R-Penalty Term
        r_term = np.log(101.0 - float(r_val))
        if r_term <= 1e-9:
            continue

        z_values = dif / r_term
        months = combined["month"].values.astype(int)

        z_results.append(pd.DataFrame({"month": months, "z": z_values}))

if not z_results:
    raise RuntimeError("No overlapping data found between pairs to compute Z-values.")

z_total_df = pd.concat(z_results, ignore_index=True)

# ==========================================
# 4. Compute Cm Table (Monthly Thresholds)
# ==========================================
print(f"[+] Calculating monthly Cm thresholds (Quantile: {QTL_LEVEL})...")
cms = []
global_median = float(z_total_df["z"].median())

for m in range(1, 13):
    z_m = z_total_df.loc[z_total_df["month"] == m, "z"].replace([np.inf, -np.inf], np.nan).dropna()
    n_pts = int(z_m.size)
    
    if n_pts == 0:
        cms.append({"month": m, "Cm": np.nan, "n_points": 0})
        continue

    # Optional upper-bound clipping (Winsorizing) for stability
    if n_pts >= 100:
        cap = np.nanquantile(z_m, 0.99)
        z_m = np.clip(z_m, None, cap)

    # Use Quantile if enough points, else Median as fallback
    cm_val = float(np.nanquantile(z_m, QTL_LEVEL)) if n_pts >= MIN_POINTS_PER_MONTH else float(np.nanmedian(z_m))
    cms.append({"month": m, "Cm": cm_val, "n_points": n_pts})

cm_df = pd.DataFrame(cms)

# Final Fallback: Interpolate missing months using neighbors or global median
if cm_df["Cm"].isna().any():
    for i, row in cm_df[cm_df["Cm"].isna()].iterrows():
        m = row["month"]
        # Look at adjacent months
        neighbors = cm_df.loc[cm_df["month"].isin([((m-2)%12)+1, (m%12)+1]) & cm_df["Cm"].notna(), "Cm"]
        cm_df.loc[i, "Cm"] = float(neighbors.mean()) if not neighbors.empty else global_median

# Save final threshold table
cm_df[["month", "Cm"]].to_csv(OUT_CM_PATH, index=False)

print(f"\n[+] Success! Monthly thresholds saved to:\n    {OUT_CM_PATH}")
print("\nCm Table Summary:")
print(cm_df)

In [ ]:
##2.2 Daily Difference 

In [ ]:
your-repo-name/
│
├── output/
│   └── advanced_qc/
│       └── cleaned_daily_data/        <-- (Input) Daily CSVs
│   └── spatial_check/             
│       ├── spatial_neighbor_pairs.csv    <-- (Input) From Step 1
│       ├── spatial_cm_table.csv          <-- (Input) From Step 2
│       ├── spatial_dif_daily.csv         <-- (Output) Daily DIF values
│       ├── spatial_t_thresholds.csv      <-- (Output) Computed T thresholds
│       └── spatial_final_flags.csv       <-- (Output) FINAL FLAGS per pair/day
└── spatial_step3_flags.py             <-- This script

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input Paths
PAIRS_PATH = BASE_DIR / "output" / "spatial_check" / "spatial_neighbor_pairs.csv"
DAILY_DIR  = BASE_DIR / "output" / "advanced_qc" / "cleaned_daily_data"
CM_TABLE_CSV = BASE_DIR / "output" / "spatial_check" / "spatial_cm_table.csv"

# Output Paths
OUT_DIR = BASE_DIR / "output" / "spatial_check"
OUT_DIF   = OUT_DIR / "spatial_dif_daily.csv"
OUT_T     = OUT_DIR / "spatial_t_thresholds.csv"
OUT_FLAGS = OUT_DIR / "spatial_final_flags.csv"

# Column Mapping
ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# Thresholds
R_MIN = 70.0              # Consistent with Step 1/2
JOINT_MIN_RAINY = 60      # Optional: Min joint rainy days filter

# ==========================================
# Utilities
# ==========================================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Extracts date from filename."""
    s = path.stem
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", s)
    if not m:
        return pd.NaT
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def normalize_id(s: pd.Series) -> pd.Series:
    """Standardizes IDs to string format."""
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

# ==========================================
# 1. Load Neighbor Pairs and Cm Table
# ==========================================
print("[+] Loading input metadata...")
pairs = pd.read_csv(PAIRS_PATH)
pairs = pairs.rename(columns={"cand_id": "cand", "aux_id": "aux", "r_score": "R"})

pairs["cand"] = normalize_id(pairs["cand"])
pairs["aux"]  = normalize_id(pairs["aux"])
pairs["R"]    = pd.to_numeric(pairs["R"], errors="coerce")

# Filter pairs based on R_MIN and Joint Rainy days
pairs_use = pairs[pairs["R"] >= R_MIN].copy()
if "joint_rainy_days" in pairs_use.columns and JOINT_MIN_RAINY is not None:
    pairs_use = pairs_use[pairs_use["joint_rainy_days"] >= JOINT_MIN_RAINY].copy()

pairs_use = pairs_use[["cand", "aux", "R"]].dropna()
if pairs_use.empty:
    raise RuntimeError("No pairs remain after R and Joint Rainy filter.")

# Load Monthly Cm values
cm_df = pd.read_csv(CM_TABLE_CSV)
cm_df["month"] = pd.to_numeric(cm_df["month"]).astype(int)
cm_map = dict(zip(cm_df["month"], cm_df["Cm"]))

# ==========================================
# 2. Load Daily Data
# ==========================================
print("[+] Loading cleaned daily rain data...")
rows = []
csv_files = sorted(DAILY_DIR.glob("*.csv"))

for fp in csv_files:
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        continue
    
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})
    df["station_id"] = normalize_id(df["station_id"])
    
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)
    
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan
    rows.append(df[["station_id", "date", "rain"]])

daily_all = pd.concat(rows, ignore_index=True).dropna(subset=["date"])
daily_all["date"] = pd.to_datetime(daily_all["date"])
daily_all["month"] = daily_all["date"].dt.month

# Standardize: Sort and remove duplicates
daily_all = daily_all.sort_values(["station_id", "date"]).drop_duplicates(subset=["station_id", "date"], keep="last")

# Create lookup dictionary
station_groups = {sid: g.set_index("date")[["rain", "month"]].sort_index() 
                  for sid, g in daily_all.groupby("station_id")}

# ==========================================
# 3. Step 2a: Compute Daily Pairwise DIF
# ==========================================
print("[+] Computing pairwise DIF values for each day...")
dif_results = []

for cand, sub_pairs in pairs_use.groupby("cand"):
    s_c = station_groups.get(cand)
    if s_c is None: continue
    
    for aux, r_val in sub_pairs[["aux", "R"]].itertuples(index=False):
        s_a = station_groups.get(aux)
        if s_a is None: continue

        # Align series
        s_a_rain = s_a[["rain"]].rename(columns={"rain": "rain_aux"})
        combined = s_c.join(s_a_rain, how="inner").dropna(subset=["rain", "rain_aux"])
        if combined.empty: continue

        ppt_c = combined["rain"].values
        ppt_a = combined["rain_aux"].values
        avg_val = 0.5 * (ppt_c + ppt_a)
        max_val = np.maximum(ppt_c, ppt_a)
        
        # DIF logic
        dif = np.where((ppt_c == 0) & (ppt_a == 0), 0.0, 
                       np.abs(ppt_c - ppt_a) / np.maximum(avg_val, 1e-12) * max_val)

        dif_results.append(pd.DataFrame({
            "cand": cand, "aux": aux, "date": combined.index,
            "month": combined["month"].values.astype(int),
            "ppt_c": ppt_c, "ppt_a": ppt_a, "dif": dif
        }))

dif_df = pd.concat(dif_results, ignore_index=True) if dif_results else pd.DataFrame()
dif_df.to_csv(OUT_DIF, index=False)
print(f" [2a] Saved: {OUT_DIF.name}")

# ==========================================
# 4. Step 2b: Compute T Threshold Grid
# ==========================================
print("[+] Computing T thresholds grid (Pair x Month)...")
month_grid = pd.DataFrame({"month": range(1, 13)})
pairs_use["_key"] = 1
month_grid["_key"] = 1
t_grid = pairs_use.merge(month_grid, on="_key").drop(columns="_key")

t_grid["cm"] = t_grid["month"].map(cm_map).astype(float)
t_grid["T"]  = t_grid["cm"] * np.log(101.0 - t_grid["R"].astype(float))

t_df = t_grid[["cand", "aux", "month", "R", "cm", "T"]].sort_values(["cand", "aux", "month"])
t_df.to_csv(OUT_T, index=False)
print(f" [2b] Saved: {OUT_T.name}")

# ==========================================
# 5. Step 2c: Generate Flags (L-Flag)
# ==========================================
print("[+] Generating spatial consistency flags (L-Flag)...")
if not dif_df.empty and not t_df.empty:
    final_flags = dif_df.merge(t_df, on=["cand", "aux", "month"], how="left")
    # L=1 means PASS (DIF < T), L=0 means SUSPECT (DIF >= T)
    final_flags["L"] = np.where(final_flags["dif"] < final_flags["T"], 1, 0).astype(int)
else:
    final_flags = pd.DataFrame()

final_flags.to_csv(OUT_FLAGS, index=False)
print(f" [2c] Final Flags Saved: {OUT_FLAGS.name}")
print("\nProcess Complete.")

In [ ]:
#3.Daily Weighted Agreement

In [ ]:
#3.Daily Weighted Agreement-Step1_Daily Labeling

In [ ]:
your-repo-name/
│
├── output/
│   └── advanced_qc/
│       └── cleaned_daily_data/        <-- (Input) Daily CSV files
│   └── spatial_check/             
│       ├── spatial_final_flags.csv    <-- (Input) Flags from Step 3
│       └── spatial_daily_labels.csv   <-- (Output) FINAL SPATIAL LABELS (Valid/Invalid)
└── spatial_step4_labeling.py          <-- This script

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input Paths
FLAGS_PATH = BASE_DIR / "output" / "spatial_check" / "spatial_final_flags.csv"
DAILY_DIR  = BASE_DIR / "output" / "advanced_qc" / "cleaned_daily_data"

# Output Path
OUT_DIR = BASE_DIR / "output" / "spatial_check"
OUT_DAILY_LABELS = OUT_DIR / "spatial_daily_labels.csv"

# Settings
ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"
R_MIN    = 70.0  # Threshold used in previous steps

# ==========================================
# 1. Load Pairwise Flags
# ==========================================
print("[+] Loading pairwise spatial flags...")
# Columns expected: cand, date, aux, R, T, dif, L
flags = pd.read_csv(FLAGS_PATH, parse_dates=["date"])

# ==========================================
# 2. Aggregate Daily Labels (Calculate Wm)
# ==========================================
print("[+] Calculating daily Wm scores and labeling...")

def aggregate_spatial_score(group):
    """
    Calculates the weighted spatial consistency index (Wm).
    Wm >= 50: Valid
    20 <= Wm < 50: Doubtful
    Wm < 20: Invalid
    """
    n_aux_used = group.shape[0]
    
    # Require at least 3 auxiliary stations for a reliable spatial check
    if n_aux_used < 3:
        return pd.Series({
            "wm_score": np.nan, 
            "spatial_label": "insufficient_neighbors", 
            "n_aux": n_aux_used
        })
    
    # Weight calculation: (R - R_min)^2
    weights = (group["R"] - R_MIN)**2
    denominator = weights.sum()
    
    if denominator <= 0:
        return pd.Series({
            "wm_score": np.nan, 
            "spatial_label": "insufficient_neighbors", 
            "n_aux": n_aux_used
        })
    
    # Weighted average of the L-flags (where L=1 is consistent, L=0 is suspect)
    numerator = (group["L"] * weights).sum()
    wm_score = 100.0 * numerator / denominator
    
    # Classification logic
    if wm_score >= 50.0:
        label = "valid"
    elif wm_score >= 20.0:
        label = "doubtful"
    else:
        label = "invalid"
        
    return pd.Series({
        "wm_score": wm_score, 
        "spatial_label": label, 
        "n_aux": n_aux_used
    })

# Apply aggregation grouped by candidate station and date
daily_labels = (flags.groupby(["cand", "date"])
                .apply(aggregate_spatial_score, include_groups=False)
                .reset_index()
                .rename(columns={"cand": "station_id"}))

# ==========================================
# 3. Attach Daily Rainfall for Final Report
# ==========================================
print("[+] Merging with original rainfall data...")
rain_data_list = []
csv_files = sorted(DAILY_DIR.glob("*.csv"))

for fp in csv_files:
    df = pd.read_csv(fp)
    if all(col in df.columns for col in [ID_COL, DATE_COL, RAIN_COL]):
        sub = df[[ID_COL, DATE_COL, RAIN_COL]].copy()
        sub = sub.rename(columns={ID_COL: "station_id", DATE_COL: "date", RAIN_COL: "ppt_mm"})
        sub["date"] = pd.to_datetime(sub["date"], errors="coerce")
        rain_data_list.append(sub)

if not rain_data_list:
    print("[!] Warning: No daily rain files found to merge.")
    final_output = daily_labels
else:
    daily_rain_all = pd.concat(rain_data_list, ignore_index=True).dropna(subset=["date"])
    
    # Clean station IDs to ensure matching
    daily_rain_all["station_id"] = daily_rain_all["station_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    daily_labels["station_id"]   = daily_labels["station_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

    # Final Merge
    final_output = (daily_labels.merge(daily_rain_all, on=["station_id", "date"], how="left")
                    .sort_values(["station_id", "date"]))

# ==========================================
# 4. Save and Summary
# ==========================================
final_output.to_csv(OUT_DAILY_LABELS, index=False)
print(f"\n[+] Success! Spatial labels saved to:\n    {OUT_DAILY_LABELS}")

# Print summary of labels
if "spatial_label" in final_output.columns:
    print("\nLabel Summary:")
    print(final_output["spatial_label"].value_counts())
    print("\nSample Output (Top 5):")
    print(final_output.head())

In [ ]:
#3.Daily Weighted Agreement-Step2_Cleaning

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input Paths
DAILY_LABELS = BASE_DIR / "output" / "spatial_check" / "spatial_daily_labels.csv"
DAILY_DIR_IN = BASE_DIR / "output" / "advanced_qc" / "cleaned_daily_data"

# Output Paths
DAILY_DIR_OUT = BASE_DIR / "output" / "advanced_qc" / "final_qc_cleaned"
DAILY_DIR_OUT.mkdir(parents=True, exist_ok=True)

# Column Mapping
ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# CLEANING POLICY (Options: "drop", "nan", "keep")
# "drop": Remove the entire row
# "nan" : Keep the row but change rainfall to NaN
# "keep": Leave data as is
POLICY_INVALID      = "drop"    
POLICY_DOUBTFUL     = "keep"    
POLICY_INSUFFICIENT = "keep"    

# ==========================================
# 1. Load Daily Spatial Labels
# ==========================================
print("[+] Loading spatial labels...")
labels_df = pd.read_csv(DAILY_LABELS, parse_dates=[DATE_COL])

# Ensure labels are filled
labels_df["spatial_label"] = labels_df["spatial_label"].fillna("insufficient_neighbors")

# ==========================================
# 2. Cleaning Function
# ==========================================
def apply_cleaning_policy(df_subset):
    """
    Applies the cleaning policy based on spatial labels.
    """
    # Merge daily data with spatial labels
    df = df_subset.merge(labels_df, on=[ID_COL, DATE_COL], how="left")
    
    # Fill missing labels (stations not processed in spatial check)
    df["spatial_label"] = df["spatial_label"].fillna("insufficient_neighbors")

    def execute_action(df_target, label_name, action):
        mask = df_target["spatial_label"] == label_name
        if action == "drop":
            return df_target.loc[~mask].copy()
        elif action == "nan":
            df_target.loc[mask, RAIN_COL] = np.nan
            return df_target
        else: # "keep"
            return df_target

    # Apply policies
    df = execute_action(df, "invalid", POLICY_INVALID)
    df = execute_action(df, "doubtful", POLICY_DOUBTFUL)
    df = execute_action(df, "insufficient_neighbors", POLICY_INSUFFICIENT)

    # Clean up columns for final output
    # Handle potentially missing columns from merge
    for c in ["wm_score", "n_aux"]:
        if c not in df.columns:
            df[c] = np.nan
            
    # Keep original column names + QC metadata
    final_cols = [ID_COL, DATE_COL, RAIN_COL, "wm_score", "spatial_label", "n_aux"]
    return df[final_cols]

# ==========================================
# 3. Process All Daily Files
# ==========================================
print(f"[+] Applying QC policy to daily files...")
all_cleaned_data = []
csv_files = sorted(DAILY_DIR_IN.glob("*.csv"))

if not csv_files:
    raise RuntimeError(f"No daily CSV files found in: {DAILY_DIR_IN}")

for fp in csv_files:
    df = pd.read_csv(fp)
    
    if any(c not in df.columns for c in [ID_COL, DATE_COL, RAIN_COL]):
        print(f" [!] Skipping {fp.name}: Required columns missing.")
        continue
        
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    
    # Process by normalized date (to handle single or multiple days per file)
    cleaned_parts = []
    for dt, group in df.groupby(df[DATE_COL].dt.normalize()):
        cleaned_batch = apply_cleaning_policy(group.copy())
        cleaned_parts.append(cleaned_batch)
        all_cleaned_data.append(cleaned_batch)
        
    if cleaned_parts:
        final_daily_df = pd.concat(cleaned_parts, ignore_index=True)
        final_daily_df.to_csv(DAILY_DIR_OUT / fp.name, index=False)
        print(f" [+] Cleaned & Saved: {fp.name}")

# ==========================================
# 4. Generate QC Summary Report
# ==========================================
print("\n[+] Generating final QC summary report...")
summary_data = pd.concat(all_cleaned_data, ignore_index=True)
summary_data["year"] = pd.to_datetime(summary_data[DATE_COL]).dt.year

# Calculate summary stats per station per year
summary = (summary_data
           .assign(
               is_valid = summary_data["spatial_label"].eq("valid").astype(int),
               has_data = summary_data[RAIN_COL].notna().astype(int)
           )
           .groupby([ID_COL, "year"])
           .agg(
               total_records=("has_data", "count"),
               valid_records=("is_valid", "sum"),
               missing_data=("has_data", lambda x: (x==0).sum())
           )
           .reset_index())

summary["validity_percentage"] = (summary["valid_records"] / summary["total_records"]) * 100.0
summary = summary.sort_values(["year", ID_COL])

SUMMARY_OUT_PATH = BASE_DIR / "output" / "spatial_check" / "relative_qc_summary_report.csv"
summary.to_csv(SUMMARY_OUT_PATH, index=False)

print(f"\n[Done] Final Cleaning Processed.")
print(f" - Final dataset: {DAILY_DIR_OUT}")
print(f" - Summary report: {SUMMARY_OUT_PATH}")
print("\nTop 10 Summary Rows:")
print(summary.head(10))

In [ ]:
#3.Daily Weighted Agreement-Step3_Station_Reporting

In [ ]:
your-repo-name/
│
├── output/
│   └── advanced_qc/
│       └── cleaned_daily_data/        <-- (Input) Files from Q-Index step
│   └── spatial_check/             
│       ├── spatial_daily_labels.csv   <-- (Input) Labels from Step 4
│       ├── summary_by_station_year.csv <-- (Output) Yearly report
│       └── summary_by_station.csv      <-- (Output) Overall station report
└── spatial_step5_qc_summary.py        <-- This script

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: Directory containing daily files after Absolute QC (Q-Index)
DAILY_BASE_DIR = BASE_DIR / "output" / "advanced_qc" / "cleaned_daily_data"

# Input: Daily labels generated from Spatial Step 4
LABELS_PATH = BASE_DIR / "output" / "spatial_check" / "spatial_daily_labels.csv"

# Output: Summary Reports
OUT_DIR = BASE_DIR / "output" / "spatial_check"
OUT_BY_STATION_YEAR = OUT_DIR / "summary_by_station_year.csv"
OUT_BY_STATION      = OUT_DIR / "summary_by_station.csv"

# Column Mapping
ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# ==========================================
# Utilities
# ==========================================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Extracts date from filename (YYYY-MM-DD)."""
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", path.stem)
    return pd.NaT if not m else pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def normalize_id(s: pd.Series) -> pd.Series:
    """Standardizes IDs to string format."""
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

def calculate_pct(df, numerator, denominator):
    """Safely calculates percentage."""
    return np.where(df[denominator] > 0, 100 * df[numerator] / df[denominator], np.nan)

# ==========================================
# 1. Load Daily Base Data
# ==========================================
print("[+] Loading daily rain data...")
base_rows = []
csv_files = sorted(DAILY_BASE_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {DAILY_BASE_DIR}")

for fp in csv_files:
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        continue

    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})
    df["station_id"] = normalize_id(df["station_id"])
    
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)
        
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan # Negative values treated as missing
    base_rows.append(df[["station_id", "date", "rain"]])

base_df = pd.concat(base_rows, ignore_index=True).dropna(subset=["date"])
base_df = base_df.sort_values(["station_id", "date"])
base_df["year"] = base_df["date"].dt.year
base_df["has_data"] = base_df["rain"].notna()

# ==========================================
# 2. Load Labels and Merge
# ==========================================
print("[+] Loading spatial labels and merging with base data...")
labels_df = pd.read_csv(LABELS_PATH, parse_dates=["date"])
labels_df["station_id"] = normalize_id(labels_df["station_id"])

# Ensure all necessary columns exist
for col in ("spatial_label", "wm_score", "n_aux"):
    if col not in labels_df.columns:
        # Fallback for different naming conventions
        if col == "spatial_label" and "label" in labels_df.columns:
            labels_df = labels_df.rename(columns={"label": "spatial_label"})
        elif col == "wm_score" and "Wm" in labels_df.columns:
            labels_df = labels_df.rename(columns={"Wm": "wm_score"})
        elif col == "n_aux" and "n_aux_used" in labels_df.columns:
            labels_df = labels_df.rename(columns={"n_aux_used": "n_aux"})
        else:
            labels_df[col] = np.nan

labels_df["spatial_label"] = labels_df["spatial_label"].str.lower()

# Left-join: Keep all station-days from base, attach labels where available
merged_df = base_df.merge(labels_df[["station_id", "date", "spatial_label", "wm_score", "n_aux"]],
                          on=["station_id", "date"], how="left")

# Fill missing metadata for days that were not processed in Spatial QC
merged_df["spatial_label"] = merged_df["spatial_label"].fillna("insufficient_neighbors")
merged_df["n_aux"] = pd.to_numeric(merged_df["n_aux"], errors="coerce").fillna(0).astype(int)

# Diagnostics
print(f" - Stations in base data: {merged_df['station_id'].nunique()}")
print(f" - Stations in labels file: {labels_df['station_id'].nunique()}")

# ==========================================
# 3. Summary by Station and Year
# ==========================================
print("[+] Generating Yearly Summary...")
# Consider only days where actual rainfall data exists
data_present = merged_df[merged_df["has_data"]].copy()

# Count occurrences of each label
counts_yr = (data_present.groupby(["station_id", "year", "spatial_label"]).size()
             .unstack("spatial_label", fill_value=0)
             .reindex(columns=["valid", "doubtful", "invalid", "insufficient_neighbors"], fill_value=0)
             .reset_index())

# Total days with data per year from base_df
total_days_yr = (merged_df.groupby(["station_id", "year"])["has_data"].sum().reset_index(name="days_total"))

summary_yr = counts_yr.merge(total_days_yr, on=["station_id", "year"], how="outer").fillna(0)
summary_yr = summary_yr.rename(columns={
    "valid": "days_valid", "doubtful": "days_doubtful",
    "invalid": "days_invalid", "insufficient_neighbors": "days_insufficient"
})

# Calculate Percentages
summary_yr["pct_valid"] = calculate_pct(summary_yr, "days_valid", "days_total")
summary_yr["pct_doubtful"] = calculate_pct(summary_yr, "days_doubtful", "days_total")
summary_yr["pct_invalid"] = calculate_pct(summary_yr, "days_invalid", "days_total")
summary_yr["pct_insufficient"] = calculate_pct(summary_yr, "days_insufficient", "days_total")

# Usable data percentage (Valid + Doubtful + Insufficient)
summary_yr["pct_usable"] = summary_yr[["pct_valid", "pct_doubtful", "pct_insufficient"]].sum(axis=1)

summary_yr.sort_values(["station_id", "year"]).to_csv(OUT_BY_STATION_YEAR, index=False)
print(f" - Saved: {OUT_BY_STATION_YEAR.name}")

# ==========================================
# 4. Summary by Station (All Years Combined)
# ==========================================
print("[+] Generating Overall Station Summary...")

counts_st = (data_present.groupby(["station_id", "spatial_label"]).size()
             .unstack("spatial_label", fill_value=0)
             .reindex(columns=["valid", "doubtful", "invalid", "insufficient_neighbors"], fill_value=0)
             .reset_index())

total_days_st = (merged_df.groupby(["station_id"])["has_data"].sum().reset_index(name="days_total"))

summary_st = counts_st.merge(total_days_st, on="station_id", how="outer").fillna(0).rename(columns={
    "valid": "days_valid", "doubtful": "days_doubtful",
    "invalid": "days_invalid", "insufficient_neighbors": "days_insufficient"
})

summary_st["pct_valid"] = calculate_pct(summary_st, "days_valid", "days_total")
summary_st["pct_doubtful"] = calculate_pct(summary_st, "days_doubtful", "days_total")
summary_st["pct_invalid"] = calculate_pct(summary_st, "days_invalid", "days_total")
summary_st["pct_insufficient"] = calculate_pct(summary_st, "days_insufficient", "days_total")
summary_st["pct_usable"] = summary_st[["pct_valid", "pct_doubtful", "pct_insufficient"]].sum(axis=1)

summary_st.sort_values("station_id").to_csv(OUT_BY_STATION, index=False)
print(f" - Saved: {OUT_BY_STATION.name}")

print("\nProcess Complete.")

In [ ]:
#3.Daily Weighted Agreement-Step4_global_synthesis

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: Directory with daily files (post-Absolute QC)
DAILY_BASE_DIR = BASE_DIR / "output" / "advanced_qc" / "cleaned_daily_data"

# Input: Daily spatial labels (Wm results)
LABELS_PATH = BASE_DIR / "output" / "spatial_check" / "spatial_daily_labels.csv"

# Output: Global summaries
OUT_DIR = BASE_DIR / "output" / "spatial_check"
OUT_LONG = OUT_DIR / "network_summary_long.csv"
OUT_WIDE = OUT_DIR / "network_summary_wide.csv"

# Column Mapping
ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# ==========================================
# Utilities
# ==========================================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Extracts date from filename (YYYY-MM-DD)."""
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", path.stem)
    return pd.NaT if not m else pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def normalize_id(s: pd.Series) -> pd.Series:
    """Standardizes IDs to string format."""
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

# ==========================================
# 1. Load Daily Base Data (Full Dataset)
# ==========================================
print("[+] Loading daily rain data for the whole network...")
base_rows = []
csv_files = sorted(DAILY_BASE_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {DAILY_BASE_DIR}")

for fp in csv_files:
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        continue
    
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})
    df["station_id"] = normalize_id(df["station_id"])
    
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)
        
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan
    base_rows.append(df[["station_id", "date", "rain"]])

base_df = pd.concat(base_rows, ignore_index=True).dropna(subset=["date"])
base_df["has_data"] = base_df["rain"].notna() # Validates presence of rain value (including 0.0)

# ==========================================
# 2. Load Labels and Merge
# ==========================================
print("[+] Merging base data with spatial labels...")
labels_df = pd.read_csv(LABELS_PATH, parse_dates=["date"])
labels_df["station_id"] = normalize_id(labels_df["station_id"])

# Standardize column naming
if "label" in labels_df.columns:
    labels_df = labels_df.rename(columns={"label": "spatial_label"})

labels_df["spatial_label"] = labels_df["spatial_label"].str.lower()

# Merge labels into the full base dataset
merged_df = base_df.merge(labels_df[["station_id", "date", "spatial_label"]],
                          on=["station_id", "date"], how="left")

# Assign 'insufficient_neighbors' to days that couldn't be checked spatially
merged_df["spatial_label"] = merged_df["spatial_label"].fillna("insufficient_neighbors")

# ==========================================
# 3. Network-Wide Statistics (Days with data only)
# ==========================================
print("[+] Computing overall statistics...")
# We only care about records where rainfall data was actually present
data_present_df = merged_df[merged_df["has_data"]].copy()

# Sort order for reporting
category_order = ["valid", "doubtful", "invalid", "insufficient_neighbors"]

# Count occurrences per category
label_counts = data_present_df["spatial_label"].value_counts().reindex(category_order, fill_value=0)
total_days_network = int(label_counts.sum())

# Format 1: Long Summary
summary_long = (pd.DataFrame({"category": category_order, "n_days": label_counts.values})
                .assign(percentage=lambda d: np.where(total_days_network > 0, 
                                                     100 * d["n_days"] / total_days_network, 
                                                     np.nan)))

# Format 2: Wide Summary (Paper-Ready)
wide_data = {"total_station_days": total_days_network}
for cat in category_order:
    n = int(label_counts[cat])
    p = (100 * n / total_days_network) if total_days_network > 0 else np.nan
    wide_data[f"{cat}_n"] = n
    wide_data[f"{cat}_pct"] = p

summary_wide = pd.DataFrame([wide_data])

# ==========================================
# 4. Save and Report
# ==========================================
summary_long.to_csv(OUT_LONG, index=False)
summary_wide.to_csv(OUT_WIDE, index=False)

print(f"\n[+] Summaries saved to:\n    - {OUT_LONG.name}\n    - {OUT_WIDE.name}")

# Print pretty console report
print("\n" + "="*45)
print("  OVERALL NETWORK QUALITY SUMMARY  ")
print("="*45)
print(f"{'Total Records:':<20} {total_days_network:,} days")
print("-" * 45)
for cat in category_order:
    n = int(label_counts[cat])
    p = (100 * n / total_days_network) if total_days_network > 0 else 0
    print(f"{cat.replace('_', ' ').capitalize():<20} {n:>12,} ({p:>6.2f}%)")
print("="*45)